In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,023 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

# Start the Ollama server as a background process
subprocess.Popen(['ollama', 'serve'])

# Wait a few seconds for the server to initialize
time.sleep(5)
print("Ollama server is running!")

Ollama server is running!


In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama run llama3.2 "Hello, how are you?"

I'm just a language model, so I don't have feelings or emotions like humans do, but thank you for asking! I'm functioning properly and ready to help with any questions or tasks you may have. How can I assist you today?



In [ ]:
import glob
import os

# Method 1: If all classes are in one parent folder
base_path = "/content/drive/MyDrive/FACMIC/data/BrainTumor/"

all_image_paths = []
for client in ['client_0','client_1','client_2','client_3']:
  client_path=os.path.join(base_path,client)
  for class_folder in ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']:
      folder_path = os.path.join(client_path, class_folder)

      # Get all jpg images
      images = glob.glob(os.path.join(folder_path, "*.jpg"))
      all_image_paths.extend(images)

      # Also get png if you have them
      images_png = glob.glob(os.path.join(folder_path, "*.png"))
      all_image_paths.extend(images_png)

print(f"Found {len(all_image_paths)} images")
print(f"First few: {all_image_paths[:3]}")

Found 3244 images
First few: ['/content/drive/MyDrive/FACMIC/data/BrainTumor/client_0/glioma_tumor/gg (231).jpg', '/content/drive/MyDrive/FACMIC/data/BrainTumor/client_0/glioma_tumor/gg (224).jpg', '/content/drive/MyDrive/FACMIC/data/BrainTumor/client_0/glioma_tumor/gg (228).jpg']


In [ ]:
!pip install ollama


In [ ]:
import ollama
import json
import os
from PIL import Image
import base64
from io import BytesIO

class LLMPromptGenerator:
    """Generate image-specific prompts using local LLM"""

    def __init__(self, model_name="llama3.2-vision"):
        self.model = model_name
        self.client = ollama.Client()

    def image_to_base64(self, image_path):
        """Convert image to base64 for LLM"""
        img = Image.open(image_path)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        return base64.b64encode(buffered.getvalue()).decode()

    def generate_prompt_for_image(self, image_path, class_name):
        """
        Generate a medical description for a single image
        """

        # Create instruction for the LLM
        instruction = f"""You are a radiologist. Look at this brain MRI image which shows {class_name.replace('_', ' ')}.
Generate a single concise medical description (one sentence, max 15 words) that describes the key radiological findings you observe.
Focus on: location, borders, enhancement pattern, surrounding tissue effects.
Format: "[your description]"
Description:"""

        try:
            # For vision models
            if "vision" in self.model:
                response = self.client.generate(
                    model=self.model,
                    prompt=instruction,
                    images=[image_path]
                )
            else:
                # For text-only models (no image, use class name)
                response = self.client.generate(
                    model=self.model,
                    prompt=f"""Generate a realistic medical description for a brain MRI showing {class_name.replace('_', ' ')}.

Examples:
- "brain MRI showing infiltrative glioma with irregular borders"
- "brain MRI showing extra-axial meningioma with dural attachment"
- "brain MRI showing sellar pituitary adenoma"

Generate one similar description (max 15 words):"""
                )

            # Extract and clean the response
            description = response['response'].strip('"')

            # Ensure it starts with "brain MRI showing"

            # if not description.lower().startswith("brain mri"):
            #     description = f"brain MRI showing {description}"

            return description

        except Exception as e:
            print(f"Error generating prompt: {e}")
            return f"brain MRI showing {class_name.replace('_', ' ')}"

    def generate_all_prompts(self, image_paths, save_path='/content/drive/MyDrive/FACMIC/llm_prompts_1.json'):
        """Generate prompts for all images"""
        prompts = {}

        print(f"Generating prompts for {len(image_paths)} images...")

        for i, img_path in enumerate(image_paths):
            class_name = os.path.basename(os.path.dirname(img_path))

            prompt = self.generate_prompt_for_image(img_path, class_name)
            prompts[img_path] = prompt

            if (i + 1) % 10 == 0:
                print(f"Progress: {i+1}/{len(image_paths)}")

        # Save prompts
        with open(save_path, 'w') as f:
            json.dump(prompts, f, indent=2)

        print(f"Saved {len(prompts)} prompts to {save_path}")
        return prompts

# Usage
generator = LLMPromptGenerator(model_name="llama3.2")

# Generate prompts for all images
# all_image_paths = [...]  # Your image paths
prompts = generator.generate_all_prompts(all_image_paths)

Generating prompts for 3244 images...
Progress: 10/3244
Progress: 20/3244
Progress: 30/3244
Progress: 40/3244
Progress: 50/3244
Progress: 60/3244
Progress: 70/3244
Progress: 80/3244
Progress: 90/3244
Progress: 100/3244
Progress: 110/3244
Progress: 120/3244
Progress: 130/3244
Progress: 140/3244
Progress: 150/3244
Progress: 160/3244
Progress: 170/3244
Progress: 180/3244
Progress: 190/3244
Progress: 200/3244
Progress: 210/3244
Progress: 220/3244
Progress: 230/3244
Progress: 240/3244
Progress: 250/3244
Progress: 260/3244
Progress: 270/3244
Progress: 280/3244
Progress: 290/3244
Progress: 300/3244
Progress: 310/3244
Progress: 320/3244
Progress: 330/3244
Progress: 340/3244
Progress: 350/3244
Progress: 360/3244
Progress: 370/3244
Progress: 380/3244
Progress: 390/3244
Progress: 400/3244
Progress: 410/3244
Progress: 420/3244
Progress: 430/3244
Progress: 440/3244
Progress: 450/3244
Progress: 460/3244
Progress: 470/3244
Progress: 480/3244
Progress: 490/3244
Progress: 500/3244
Progress: 510/3244
Pr